In [1]:
import pandas as pd
import numpy as np
import json

from pathlib import Path
from sklearn.model_selection import train_test_split

In [2]:
DATA_PATH = Path("../data/PhiUSIIL_Phishing_URL_Dataset.csv")

df = pd.read_csv(DATA_PATH)

print("Dimensi dataset:", df.shape)

df.head()

Dimensi dataset: (235795, 56)


,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,...,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,...,1,0,1,244,15,34,72,1,85,1


In [3]:
print("Missing values:", df.isnull().sum().sum())
print("Duplikasi baris:", df.duplicated().sum())
print("Duplikasi URL:", df['URL'].duplicated().sum())

Missing values: 0
Duplikasi baris: 0
Duplikasi URL: 425


In [4]:
conflicting_urls = (
    df.groupby('URL')['label'].nunique() > 1
).sum()

print("URL dengan label bertentangan:", conflicting_urls)

assert conflicting_urls == 0, "Terdapat konflik label URL"

df_clean = df.drop_duplicates(
    subset=['URL'],
    keep='first'
).reset_index(drop=True)

print("Dimensi sebelum preprocessing:", df.shape)
print("Dimensi setelah preprocessing:", df_clean.shape)

URL dengan label bertentangan: 0
Dimensi sebelum preprocessing: (235795, 56)
Dimensi setelah preprocessing: (235370, 56)


In [5]:
excluded_columns = [
    'FILENAME',
    'URL',
    'Domain',
    'TLD',
    'Title',
    'label'
]

suspected_leakage = [
    'URLSimilarityIndex',
    'TLDLegitimateProb',
    'URLCharProb'
]

In [6]:
full_features = (
    df_clean
    .drop(columns=excluded_columns + suspected_leakage)
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

print("Jumlah full features:", len(full_features))

full_features

Jumlah full features: 47


['URLLength',
 'DomainLength',
 'IsDomainIP',
 'CharContinuationRate',
 'TLDLength',
 'NoOfSubDomain',
 'HasObfuscation',
 'NoOfObfuscatedChar',
 'ObfuscationRatio',
 'NoOfLettersInURL',
 'LetterRatioInURL',
 'NoOfDegitsInURL',
 'DegitRatioInURL',
 'NoOfEqualsInURL',
 'NoOfQMarkInURL',
 'NoOfAmpersandInURL',
 'NoOfOtherSpecialCharsInURL',
 'SpacialCharRatioInURL',
 'IsHTTPS',
 'LineOfCode',
 'LargestLineLength',
 'HasTitle',
 'DomainTitleMatchScore',
 'URLTitleMatchScore',
 'HasFavicon',
 'Robots',
 'IsResponsive',
 'NoOfURLRedirect',
 'NoOfSelfRedirect',
 'HasDescription',
 'NoOfPopup',
 'NoOfiFrame',
 'HasExternalFormSubmit',
 'HasSocialNet',
 'HasSubmitButton',
 'HasHiddenFields',
 'HasPasswordField',
 'Bank',
 'Pay',
 'Crypto',
 'HasCopyrightInfo',
 'NoOfImage',
 'NoOfCSS',
 'NoOfJS',
 'NoOfSelfRef',
 'NoOfEmptyRef',
 'NoOfExternalRef']

In [7]:
url_only_features = [
    'URLLength',
    'DomainLength',
    'IsDomainIP',
    'TLDLength',
    'NoOfSubDomain',
    'HasObfuscation',
    'NoOfObfuscatedChar',
    'ObfuscationRatio',
    'NoOfLettersInURL',
    'LetterRatioInURL',
    'NoOfDegitsInURL',
    'DegitRatioInURL',
    'NoOfEqualsInURL',
    'NoOfQMarkInURL',
    'NoOfAmpersandInURL',
    'NoOfOtherSpecialCharsInURL',
    'SpacialCharRatioInURL',
    'IsHTTPS',
    'CharContinuationRate'
]

assert set(url_only_features).issubset(full_features)

print("Jumlah URL-only features:", len(url_only_features))

url_only_features

Jumlah URL-only features: 19


['URLLength',
 'DomainLength',
 'IsDomainIP',
 'TLDLength',
 'NoOfSubDomain',
 'HasObfuscation',
 'NoOfObfuscatedChar',
 'ObfuscationRatio',
 'NoOfLettersInURL',
 'LetterRatioInURL',
 'NoOfDegitsInURL',
 'DegitRatioInURL',
 'NoOfEqualsInURL',
 'NoOfQMarkInURL',
 'NoOfAmpersandInURL',
 'NoOfOtherSpecialCharsInURL',
 'SpacialCharRatioInURL',
 'IsHTTPS',
 'CharContinuationRate']

In [8]:
feature_comparison = pd.DataFrame({
    'Jenis Fitur': [
        'Full Features',
        'URL-Only Features'
    ],
    'Jumlah Fitur': [
        len(full_features),
        len(url_only_features)
    ]
})

feature_comparison

,Jenis Fitur,Jumlah Fitur
0,Full Features,47
1,URL-Only Features,19


In [9]:
X_full = df_clean[full_features]

X_url = df_clean[url_only_features]

y = df_clean['label'].map({
    0: 1,
    1: 0
})

groups = df_clean['Domain']

print("Dimensi X Full:", X_full.shape)
print("Dimensi X URL-Only:", X_url.shape)
print("Dimensi y:", y.shape)
print("Dimensi groups:", groups.shape)

Dimensi X Full: (235370, 47)
Dimensi X URL-Only: (235370, 19)
Dimensi y: (235370,)
Dimensi groups: (235370,)


In [10]:
train_idx, test_idx = train_test_split(
    df_clean.index.to_numpy(),
    test_size=0.3,
    random_state=42,
    stratify=y
)

X_train_full = X_full.loc[train_idx]
X_test_full = X_full.loc[test_idx]

X_train_url = X_url.loc[train_idx]
X_test_url = X_url.loc[test_idx]

y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

In [11]:
print("Full Features")
print("Training:", X_train_full.shape)
print("Testing:", X_test_full.shape)

print("\nURL-Only Features")
print("Training:", X_train_url.shape)
print("Testing:", X_test_url.shape)

print("\nTarget")
print("Training:", y_train.shape)
print("Testing:", y_test.shape)

Full Features
Training: (164759, 47)
Testing: (70611, 47)

URL-Only Features
Training: (164759, 19)
Testing: (70611, 19)

Target
Training: (164759,)
Testing: (70611,)


In [12]:
train_domains = set(groups.loc[train_idx])
test_domains = set(groups.loc[test_idx])

overlap = train_domains.intersection(test_domains)

print("Jumlah domain training:", len(train_domains))
print("Jumlah domain testing:", len(test_domains))
print("Jumlah domain overlap:", len(overlap))

print(
    "Persentase overlap:",
    round(len(overlap) / len(test_domains) * 100, 2),
    "%"
)

Jumlah domain training: 155235
Jumlah domain testing: 67484
Jumlah domain overlap: 2633
Persentase overlap: 3.9 %


In [13]:
DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")

DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

df_clean.to_csv(
    DATA_DIR / "PhiUSIIL_Preprocessed.csv",
    index=False
)


In [14]:
feature_config = {
    "full_features": full_features,
    "url_only_features": url_only_features,
    "excluded_columns": excluded_columns,
    "suspected_leakage": suspected_leakage,
    "random_state": 42,
    "test_size": 0.3,
    "positive_class": "Phishing"
}

with open(
    RESULTS_DIR / "feature_config.json",
    "w"
) as f:
    json.dump(feature_config, f, indent=4)

np.savez(
    RESULTS_DIR / "baseline_split.npz",
    train_idx=train_idx,
    test_idx=test_idx
)

print("Hasil preprocessing berhasil disimpan.")

Hasil preprocessing berhasil disimpan.
